# Read pickel for Feature Engineering

In [77]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [78]:
df = pd.read_pickle('../pickle/itsm_clean.pkl')
df.head()

,CI_Name,CI_Cat,CI_Subcat,WBS,Incident_ID,Status,Impact,Urgency,Priority,number_cnt,Category,KB_number,Alert_Status,No_of_Reassignments,Open_Time,Resolved_Time,Close_Time,Handle_Time_hrs,Closure_Code,No_of_Related_Interactions,Related_Interaction,is_high_priority,YearMonth
0,SUB000508,subapplication,Web Based Application,WBS000162,IM0000004,Closed,4,4,4.0,0.601292,incident,KM0000553,closed,26.0,2012-02-05 13:32:00,2013-11-04 13:50:00,2013-11-04 13:51:00,15312.316667,Other,1.0,SD0000007,0,2012-02
1,WBA000124,application,Web Based Application,WBS000088,IM0000005,Closed,3,3,3.0,0.415050,incident,KM0000611,closed,33.0,2012-03-12 15:44:00,2013-12-02 12:36:00,2013-12-02 12:36:00,15116.866667,Software,1.0,SD0000011,0,2012-03
2,WBA000124,application,Web Based Application,WBS000088,IM0000011,Closed,4,4,4.0,0.642927,incident,KM0000611,closed,13.0,2012-07-17 11:49:00,2013-11-14 09:31:00,2013-11-14 09:31:00,11637.700000,Operator error,1.0,SD0000025,0,2012-07
3,WBA000124,application,Web Based Application,WBS000088,IM0000012,Closed,4,4,4.0,0.345258,incident,KM0000611,closed,2.0,2012-08-10 11:01:00,2013-11-08 13:55:00,2013-11-08 13:55:00,10922.900000,Other,1.0,SD0000029,0,2012-08
4,WBA000124,application,Web Based Application,WBS000088,IM0000013,Closed,4,4,4.0,0.006676,incident,KM0000611,closed,4.0,2012-08-10 11:27:00,2013-11-08 13:54:00,2013-11-08 13:54:00,10922.450000,Other,1.0,SD0000031,0,2012-08


In [79]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45117 entries, 0 to 45116
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CI_Name                     45117 non-null  str           
 1   CI_Cat                      45117 non-null  str           
 2   CI_Subcat                   45117 non-null  str           
 3   WBS                         45117 non-null  str           
 4   Incident_ID                 45117 non-null  str           
 5   Status                      45117 non-null  str           
 6   Impact                      45117 non-null  int64         
 7   Urgency                     45117 non-null  int64         
 8   Priority                    45117 non-null  float64       
 9   number_cnt                  45117 non-null  float64       
 10  Category                    45117 non-null  str           
 11  KB_number                   45117 non-null  str           
 12  A

## Drop column to prevent Data leakage

In [80]:
df = df.drop(columns=['CI_Name', 'WBS', 'Incident_ID', 'KB_number','Related_Interaction', 'Alert_Status','Impact', 'Urgency', 'Priority', 'YearMonth'])

In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45117 entries, 0 to 45116
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CI_Cat                      45117 non-null  str           
 1   CI_Subcat                   45117 non-null  str           
 2   Status                      45117 non-null  str           
 3   number_cnt                  45117 non-null  float64       
 4   Category                    45117 non-null  str           
 5   No_of_Reassignments         45117 non-null  float64       
 6   Open_Time                   45117 non-null  datetime64[us]
 7   Resolved_Time               43483 non-null  datetime64[us]
 8   Close_Time                  45117 non-null  datetime64[us]
 9   Handle_Time_hrs             45117 non-null  float64       
 10  Closure_Code                45117 non-null  str           
 11  No_of_Related_Interactions  45117 non-null  float64       
 12  i

In [82]:
df.shape

(45117, 13)

## Convert CI_Subcat from 64 unique values into the top 15 values and group the remaining values as Other

In [83]:
top_15_subcat = df['CI_Subcat'].value_counts().head(15).index.tolist()

print(top_15_subcat)

['Server Based Application', 'Web Based Application', 'Desktop Application', 'Laptop', 'SAP', 'Banking Device', 'Citrix', 'SAN', 'Client Based Application', 'Desktop', 'DataCenterEquipment', 'System Software', 'Monitor', 'Database', 'Windows Server']


In [84]:
def check_subcat(x):
    if x in top_15_subcat:
        return x
    else:
        return 'other'


df['CI_Subcat'] = df['CI_Subcat'].apply(check_subcat)

## Retrive data from Oper_date_time into Open_month,open_day_of_week and Open_hour

In [85]:
df['open_month'] = df['Open_Time'].dt.month
df['open_dayofweek'] = df['Open_Time'].dt.day_name()
df['open_hour'] = df['Open_Time'].dt.hour

# Drop columns

df = df.drop(columns=['Open_Time','Resolved_Time','Close_Time'])

## One-hotEncoding

In [86]:
df = pd.get_dummies(df,columns=['CI_Cat', 'CI_Subcat', 'Category', 'Status', 'Closure_Code','open_dayofweek'])

In [87]:
df.head()

,number_cnt,No_of_Reassignments,Handle_Time_hrs,No_of_Related_Interactions,is_high_priority,open_month,open_hour,CI_Cat_Phone,CI_Cat_application,CI_Cat_applicationcomponent,CI_Cat_computer,CI_Cat_database,CI_Cat_displaydevice,CI_Cat_hardware,CI_Cat_networkcomponents,CI_Cat_officeelectronics,CI_Cat_software,CI_Cat_storage,CI_Cat_subapplication,CI_Subcat_Banking Device,CI_Subcat_Citrix,CI_Subcat_Client Based Application,CI_Subcat_DataCenterEquipment,CI_Subcat_Database,CI_Subcat_Desktop,CI_Subcat_Desktop Application,CI_Subcat_Laptop,CI_Subcat_Monitor,CI_Subcat_SAN,CI_Subcat_SAP,CI_Subcat_Server Based Application,CI_Subcat_System Software,CI_Subcat_Web Based Application,CI_Subcat_Windows Server,CI_Subcat_other,Category_complaint,Category_incident,Category_request for information,Status_Closed,Status_Work in progress,Closure_Code_Data,Closure_Code_Hardware,Closure_Code_Inquiry,Closure_Code_Kwaliteit van de output,Closure_Code_No error - works as designed,Closure_Code_Operator error,Closure_Code_Other,Closure_Code_Overig,Closure_Code_Questions,Closure_Code_Referred,Closure_Code_Software,Closure_Code_Unknown,Closure_Code_User error,Closure_Code_User manual not used,open_dayofweek_Friday,open_dayofweek_Monday,open_dayofweek_Saturday,open_dayofweek_Sunday,open_dayofweek_Thursday,open_dayofweek_Tuesday,open_dayofweek_Wednesday
0,0.601292,26.0,15312.316667,1.0,0,2,13,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False
1,0.415050,33.0,15116.866667,1.0,0,3,15,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False
2,0.642927,13.0,11637.700000,1.0,0,7,11,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
3,0.345258,2.0,10922.900000,1.0,0,8,11,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False
4,0.006676,4.0,10922.450000,1.0,0,8,11,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False


## Feature Engineering — Observation Report

### Overview
Building on the cleaned dataset (45,117 records) and EDA findings, feature engineering was performed to transform the data into a model-ready format. This involved removing irrelevant and leakage-prone columns, extracting new features from datetime fields, and encoding categorical variables.

---

### 1. Column Removal

The following columns were dropped prior to encoding:

| Column(s) | Reason |
|---|---|
| `CI_Name`, `WBS`, `Incident_ID`, `KB_number`, `Related_Interaction` | High-cardinality identifier columns with no generalizable predictive pattern |
| `Alert_Status` | Zero variance (single unique value across all records) |
| `Impact`, `Urgency`, `Priority` | Excluded to prevent **data leakage** — `Priority` is directly derived from `Impact` and `Urgency` via the Priority Matrix, and the target variable (`is_high_priority`) is in turn derived from `Priority`. Including these would allow the model to reverse-engineer the target rather than learn genuine predictive patterns. |
| `YearMonth` | Helper column created only for EDA time-trend visualization; not needed as a model feature |

---

### 2. Datetime Feature Extraction

Rather than dropping datetime information entirely, three new features were derived from `Open_Time` (the only timestamp available at the moment a new ticket is created, and therefore safe to use without leakage):

- `open_month` (1–12) — retained as a numeric feature to capture potential seasonal patterns.
- `open_hour` (0–23) — retained as a numeric feature to capture time-of-day patterns (e.g., business hours vs. after-hours).
- `open_dayofweek` (Monday–Sunday) — extracted as a categorical feature and one-hot encoded, since day-of-week is cyclical and has no meaningful numeric order.

`Resolved_Time` and `Close_Time` were **dropped without feature extraction**, since these timestamps only exist after a ticket is resolved/closed — using them (or any derived features) would constitute data leakage, as this information is unavailable at prediction time for a new incoming ticket.

---

### 3. Categorical Encoding

The following categorical columns were one-hot encoded using `pd.get_dummies()`: `CI_Cat`, `CI_Subcat`, `Category`, `Status`, `Closure_Code`, and `open_dayofweek`.

**Special handling for `CI_Subcat`:** This column originally contained 62 unique values, most of which were rare occurrences (long-tail distribution, as observed during EDA). To avoid excessive dimensionality from encoding, only the **top 15 most frequent sub-categories** (covering the vast majority of records — e.g., "Server Based Application" and "Web Based Application" alone account for over 33,000 of 45,117 tickets) were retained individually; all remaining sub-categories were grouped into a single `"Other"` category before encoding.

---

### 4. Final Feature Set

After column removal, feature extraction, and encoding, the dataset expanded from its original set of columns to **50 columns** (45,117 rows unchanged). All columns are now in numeric or boolean (True/False) format, with no remaining text/object columns, identifiers, or leakage-prone fields.

**Final feature categories:**
- Numeric: `number_cnt`, `No_of_Reassignments`, `Handle_Time_hrs`, `No_of_Related_Interactions`, `open_month`, `open_hour`
- One-hot encoded: `CI_Cat_*`, `CI_Subcat_*`, `Category_*`, `Status_*`, `Closure_Code_*`, `open_dayofweek_*`
- Target: `is_high_priority`

---

### Next Steps
- Save the final feature-engineered dataset (pickle format, to preserve data types).
- Proceed to modeling: separate features (X) from the target (y), perform a stratified train-test split, and address class imbalance (98.46% vs. 1.53%) using `class_weight='balanced'` and/or SMOTE.

In [88]:
df.to_pickle('../pickle/itsm_features.pkl')